# LLM Classification Finetuning — v0.2 Llama-3.2-3B LoRA (TPU v5e-8)

**Approach:** Llama-3.2-3B-Instruct fine-tuned with LoRA on Kaggle TPU v5e-8.
Label-token readout at inference (softmax over A/B/tie logits) → 3-class probabilities for log loss.

**Strategy:**
- 5-fold stratified split; train on 4 folds, eval on held-out fold
- A/B swap augmentation to fight position bias
- Swap-TTA at inference: average (A,B) + (B,A) predictions
- Accelerator: TPU v5e-8 (must be selected manually in Kaggle UI after push)

**Why TPU over T4×2:** T4×2 with SFTTrainer falls back to `nn.DataParallel`, which is
incompatible with TRL's `_chunked_cross_entropy_loss`. True DDP requires `torchrun` —
impossible inside a Kaggle notebook. TPU v5e-8 (8 chips × 16 GB = 128 GB) handles
multi-chip distribution via XLA without this limitation.

**Model source:** `metaresearch/llama-3.2/transformers/3b-instruct/1` from Kaggle Models.

---

> **⚠ PUSH WORKFLOW — KAGGLE ALWAYS AUTO-STARTS ON PUSH:**
> 1. `zsh scripts/push_notebook.sh v0.2-llama-qlora`
> 2. **Immediately open the Kaggle UI and STOP the auto-started run** (there is no metadata flag to prevent this)
> 3. Accelerator → TPU v5e-8 → Save Version → Save & Run All

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.44', 'peft>=0.12', 'trl>=0.10',
    'accelerate>=0.33', 'datasets>=2.20',
], check=True)

print('Dependencies installed')

In [ ]:
# ── Cell 2: Imports & device detection ────────────────────────────────────────
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

try:
    import torch_xla.core.xla_model as xm
    DEVICE_TYPE = 'tpu'
    print(f'TPU detected: {xm.xla_device()}')
except ImportError:
    DEVICE_TYPE = 'cuda' if torch.cuda.is_available() else 'cpu'
    if DEVICE_TYPE == 'cuda':
        for i in range(torch.cuda.device_count()):
            print(f'  cuda:{i} — {torch.cuda.get_device_name(i)}')
    else:
        print('CPU only')

print(f'Device type : {DEVICE_TYPE}')
print('Imports OK')

In [ ]:
# ── Cell 3: Path discovery & hyperparameters ──────────────────────────────────
import os
from pathlib import Path

OUTPUT = Path('/kaggle/working')

# Locate competition data
_data_candidates = [Path('/kaggle/input/llm-classification-finetuning')]
INPUT = next((p for p in _data_candidates if (p / 'train.csv').exists()), None)
if INPUT is None:
    _found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if _found:
        INPUT = Path(_found[0]).parent

# Locate Llama model (mounted from Kaggle Models)
_llama_configs = [
    p for p in glob.glob('/kaggle/input/**/config.json', recursive=True)
    if 'llama' in p.lower()
]
MODEL_PATH = str(Path(_llama_configs[0]).parent) if _llama_configs else None

print(f'Accelerator : {DEVICE_TYPE}')
print(f'Input path  : {INPUT}')
print(f'Model path  : {MODEL_PATH}')

if MODEL_PATH is None:
    print('Available /kaggle/input/ entries:')
    for p in sorted(Path('/kaggle/input').rglob('config.json')):
        print(f'  {p}')

assert INPUT is not None, 'Competition data not found'
assert MODEL_PATH is not None, 'Llama model not found — ensure model_sources contains metaresearch/llama-3.2/transformers/3b-instruct/1'

# Hyperparameters
N_FOLDS      = 5
TRAIN_FOLD   = 0
MAX_SEQ_LEN  = 1024
BATCH_SIZE   = 2      # per chip; 8 chips → 16 effective on TPU v5e-8
GRAD_ACCUM   = 8
NUM_EPOCHS   = 1
LR           = 2e-4
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
SEED         = 42
LABEL_TOKENS = ['A', 'B', 'tie']

# Pipeline validation cap — set to None to use all data (for v0.3+ production runs)
MAX_TRAIN_SAMPLES = 50_000

print('Config OK')

In [ ]:
# ── Cell 4: Load data ─────────────────────────────────────────────────────────
train = pd.read_csv(INPUT / 'train.csv')
test  = pd.read_csv(INPUT / 'test.csv')

winner_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
train['label'] = train[winner_cols].values.argmax(axis=1).astype(int)
train['label_str'] = train['label'].map({0: 'A', 1: 'B', 2: 'tie'})

print(f'Train: {train.shape}  Test: {test.shape}')
print(train['label_str'].value_counts())

In [ ]:
# ── Cell 5: Prompt builder ────────────────────────────────────────────────────
SYSTEM = (
    'You are an expert judge evaluating AI assistant responses. '
    'Given a user prompt and two responses (A and B), output which response '
    'a human would prefer: A, B, or tie.'
)

def build_prompt(row, swap=False):
    ra = str(row['response_a'] or '')
    rb = str(row['response_b'] or '')
    if swap:
        ra, rb = rb, ra
    prompt = str(row['prompt'] or '')
    max_resp_chars = MAX_SEQ_LEN * 3
    return (
        f'PROMPT:\n{prompt}\n\n'
        f'RESPONSE A:\n{ra[:max_resp_chars]}\n\n'
        f'RESPONSE B:\n{rb[:max_resp_chars]}\n\n'
        f'Which response is better? Answer with only: A, B, or tie.'
    )

def label_for_swap(label_str, swap):
    if not swap or label_str == 'tie':
        return label_str
    return 'B' if label_str == 'A' else 'A'

print('Prompt builder ready')

In [ ]:
# ── Cell 6: Tokenizer + fold split + swap-augmented training set ──────────────
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.model_max_length = MAX_SEQ_LEN
print(f'Tokenizer loaded from {MODEL_PATH}')

def make_chat_text(row, swap=False):
    messages = [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': build_prompt(row, swap)},
        {'role': 'assistant', 'content': label_for_swap(row['label_str'], swap)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(train, train['label']))
tr_idx, val_idx = splits[TRAIN_FOLD]

df_tr  = train.iloc[tr_idx].reset_index(drop=True)
df_val = train.iloc[val_idx].reset_index(drop=True)

train_texts = []
for _, row in df_tr.iterrows():
    train_texts.append(make_chat_text(row, swap=False))
    train_texts.append(make_chat_text(row, swap=True))

print(f'Fold {TRAIN_FOLD}: train={len(df_tr)} ({len(train_texts)} w/ swap aug)  val={len(df_val)}')

# Pipeline validation cap — subsample before tokenization to speed up the run.
# Set MAX_TRAIN_SAMPLES = None in cell-3 to use all data for production runs.
if MAX_TRAIN_SAMPLES and len(train_texts) > MAX_TRAIN_SAMPLES:
    import random; random.seed(SEED)
    train_texts = random.sample(train_texts, MAX_TRAIN_SAMPLES)
    print(f'Capped to {len(train_texts):,} samples for pipeline validation')

print('Pre-tokenizing dataset (parallel) …')

# Pre-tokenize on CPU before training cell so the TPU is not idle during prep.
# SFTTrainer's internal single-threaded tokenization (~2h for 91k examples)
# triggers Kaggle's 2h TPU idle timeout — see docs/investigate/2026-06-27-v02-t4x2-ddp.md §2.
raw_ds = Dataset.from_dict({'text': train_texts})

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

train_ds = raw_ds.map(
    tokenize_fn,
    batched=True,
    batch_size=1000,
    num_proc=4,
    remove_columns=['text'],
    desc='Tokenizing',
)
print(f'Tokenization complete — {len(train_ds):,} examples ready')

In [ ]:
# ── Cell 7: Load model with LoRA (bf16, TPU — no device_map) ─────────────────
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

# Load to CPU — XLA moves tensors to TPU chips automatically.
# bf16 required on TPU (fp16 unsupported by XLA).
# attn_implementation='eager' required: flash/sdpa not supported on TPU XLA.
# No device_map — 'auto' triggers GPU model parallelism, incompatible with SFTTrainer.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
    local_files_only=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── Cell 8: Train ─────────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

assert DEVICE_TYPE == 'tpu', (
    f'Training requires TPU v5e-8 (got {DEVICE_TYPE!r}). '
    'Stop this auto-run in the Kaggle UI → Accelerator → TPU v5e-8 → Save & Run All.'
)

output_dir = OUTPUT / f'fold{TRAIN_FOLD}-adapter'

sft_config = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    dataloader_drop_last=True,
    logging_steps=25,
    save_strategy='epoch',
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    seed=SEED,
    report_to='none',
    # dataset is pre-tokenized in cell-6 — skip SFTTrainer's internal tokenization
    dataset_text_field='',
    max_length=MAX_SEQ_LEN,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(str(output_dir))
print(f'Adapter saved → {output_dir}')

In [ ]:
# ── Cell 9: Inference helper (label-token readout) ────────────────────────────
label_token_ids = [
    tokenizer.encode(t, add_special_tokens=False)[0]
    for t in LABEL_TOKENS
]
print(f'Label token ids: {dict(zip(LABEL_TOKENS, label_token_ids))}')

# Device-agnostic — works after DDP training on any rank config
infer_device = next(model.parameters()).device
print(f'Inference device: {infer_device}')

def predict_proba(df, batch_size=4, swap=False):
    model.eval()
    all_probs = []
    rows = [r for _, r in df.iterrows()]
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        texts = [
            tokenizer.apply_chat_template(
                [{'role': 'system', 'content': SYSTEM},
                 {'role': 'user',   'content': build_prompt(row, swap=swap)}],
                tokenize=False, add_generation_prompt=True
            )
            for row in batch
        ]
        enc = tokenizer(
            texts, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_SEQ_LEN
        ).to(infer_device)
        with torch.no_grad():
            last_logits = model(**enc).logits[:, -1, :]
        probs = torch.softmax(
            last_logits[:, label_token_ids].float(), dim=-1
        ).cpu().numpy()
        all_probs.append(probs)
        if (i // batch_size) % 50 == 0:
            print(f'  {i+len(batch)}/{len(rows)}')
    return np.vstack(all_probs)

print('Inference helper ready')

In [ ]:
# ── Cell 10: OOF evaluation ───────────────────────────────────────────────────
print('OOF inference (swap-TTA) …')
probs_fwd = predict_proba(df_val, swap=False)
probs_swp = predict_proba(df_val, swap=True)
oof_probs = (probs_fwd + probs_swp[:, [1, 0, 2]]) / 2

oof_ll = log_loss(df_val['label'].values, oof_probs)
print(f'Fold {TRAIN_FOLD} OOF log loss (swap-TTA): {oof_ll:.4f}')

In [ ]:
# ── Cell 11: Test inference & submission ──────────────────────────────────────
print('Test inference (swap-TTA) …')
test_fwd = predict_proba(test, swap=False)
test_swp = predict_proba(test, swap=True)
test_probs = (test_fwd + test_swp[:, [1, 0, 2]]) / 2

sub = pd.DataFrame({
    'id':             test['id'],
    'winner_model_a': test_probs[:, 0],
    'winner_model_b': test_probs[:, 1],
    'winner_tie':     test_probs[:, 2],
})
sub.to_csv(OUTPUT / 'submission.csv', index=False)

print(f'submission.csv written — {len(sub):,} rows')
print(sub.head())
print(f'Prob sums: {test_probs.sum(axis=1)[:5].round(4)}')
print(f'OOF fold {TRAIN_FOLD}: {oof_ll:.4f}  (v0.1 baseline: 1.0404)')